# Inference — Predict Test Set
Load model .pt, predict 1458 test images, save submission.csv

In [1]:
import sys, json
sys.path.insert(0, "../..")

import torch
import pandas as pd
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from tqdm import tqdm

from modules.utils.config import *
from modules.models.factory import TrashClassifier
from modules.utils.load_data import load_test

/home/prayudahlah/projects/external/big-data-big-trouble/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
MODEL_NAME = "yuda_convnext_tiny"       # nama file .pt tanpa ekstensi
MODEL_PATH = RESULTS / f"{MODEL_NAME}.pt"
JSON_PATH  = RESULTS / f"{MODEL_NAME}.json"
OUTPUT_CSV = PROJECT_ROOT / f"submission_{MODEL_NAME}.csv"

BATCH_SIZE = 32
USE_TTA = False                          # Test Time Augmentation (lebih lambat, +0.005-0.015)
print(f"Model: {MODEL_PATH}")
print(f"Output: {OUTPUT_CSV}")

Model: /home/prayudahlah/projects/external/big-data-big-trouble/notebooks/experiments/results/yuda_convnext_tiny.pt
Output: /home/prayudahlah/projects/external/big-data-big-trouble/submission_yuda_convnext_tiny.csv


In [3]:
with open(JSON_PATH) as f:
    meta = json.load(f)

model = TrashClassifier(meta["encoder_name"], num_classes=NUM_CLASSES).to(DEVICE)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE, weights_only=True))
model.eval()

print(f"Loaded: {meta['name']}")
print(f"Encoder: {meta['encoder_name']}")
print(f"Val F1:  {meta['best_val_f1']:.4f}")

Loaded: yuda_convnext_tiny
Encoder: convnext_tiny
Val F1:  0.9742


In [4]:
test_df = load_test()
print(f"Test images: {len(test_df)}")

tensor_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

tta_transforms = []
if USE_TTA:
    tta_transforms = [
        lambda img: transforms.functional.resize(img, (IMG_SIZE, IMG_SIZE)),
        lambda img: transforms.functional.resize(img, (IMG_SIZE, IMG_SIZE)) if True else img,
    ]

class TestDataset(Dataset):
    def __init__(self, df):
        self.df = df
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(PROJECT_ROOT / row["path"]).convert("RGB")
        img = img.resize((IMG_SIZE, IMG_SIZE))
        return tensor_transform(img), int(row["image_id"])

loader = DataLoader(
    TestDataset(test_df),
    batch_size=BATCH_SIZE, shuffle=False,
    num_workers=4, pin_memory=True,
)
print(f"Batches: {len(loader)}")

Test images: 1458
Batches: 46


In [5]:
all_ids, all_preds = [], []

with torch.no_grad():
    for images, ids in tqdm(loader, desc="Inference"):
        images = images.to(DEVICE)
        outputs = model(images)

        if USE_TTA:
            probs = torch.softmax(outputs, dim=1)
            for flip in [False, True]:
                if flip:
                    flip_img = transforms.functional.hflip(images)
                    outs = model(flip_img)
                    probs += torch.softmax(outs, dim=1)
            probs /= (1 + len(tta_transforms))
            preds = probs.argmax(dim=1).cpu().numpy()
        else:
            preds = outputs.argmax(dim=1).cpu().numpy()

        all_ids.extend(ids.numpy())
        all_preds.extend(preds)

Inference: 100%|██████████| 46/46 [00:06<00:00,  7.11it/s]


In [6]:
label_names = [CLASS_LABELS[p] for p in all_preds]
df = pd.DataFrame({"id": all_ids, "predicted": label_names})
df.to_csv(OUTPUT_CSV, index=False)
print(f"Saved: {OUTPUT_CSV}")
display(df.head(10))
print(f"\nDistribusi prediksi:")
display(df["predicted"].value_counts())

Saved: /home/prayudahlah/projects/external/big-data-big-trouble/submission_yuda_convnext_tiny.csv


,id,predicted
0,1,2_Organic
1,10,2_Organic
2,100,0_Recyclable
3,1000,2_Organic
4,1001,0_Recyclable
5,1002,2_Organic
6,1003,0_Recyclable
7,1004,0_Recyclable
8,1005,1_Electronic
9,1006,2_Organic



Distribusi prediksi:


predicted
2_Organic       759
0_Recyclable    495
1_Electronic    204
Name: count, dtype: int64